# Python Context Managers

A practical notebook on context managers using built-in, class-based, and function-based patterns.

## 1. Import Setup and Sample Resources

Import standard modules and create temporary sample resources used in later sections.

In [1]:
import contextlib
import io
import tempfile
import time
from pathlib import Path

# Shared temporary workspace for this notebook.
WORKSPACE = Path(tempfile.mkdtemp(prefix='cm_demo_'))
SAMPLE_FILE = WORKSPACE / 'sample.txt'
SAMPLE_FILE.write_text('line-1\nline-2\n', encoding='utf-8')

# In-memory stream resource.
MEM_STREAM = io.StringIO('alpha,beta,gamma')

print('Workspace:', WORKSPACE)
print('Sample file exists:', SAMPLE_FILE.exists())

Workspace: /tmp/cm_demo_e_1e92nk
Sample file exists: True


## 2. Use Built-in Context Managers with `with`

Use `with open(...) as f:` for automatic open/close handling.

In [2]:
out_file = WORKSPACE / 'built_in_write.txt'

with out_file.open('w', encoding='utf-8') as f:
    f.write('hello from with\n')
    was_closed_inside = f.closed

with out_file.open('r', encoding='utf-8') as f:
    content = f.read().strip()

is_closed_after = f.closed

print('Closed inside block:', was_closed_inside)
print('Closed after block:', is_closed_after)
print('Read content:', content)

Closed inside block: False
Closed after block: True
Read content: hello from with


## 3. Build a Class-Based Context Manager (`__enter__` / `__exit__`)

A custom class can acquire a resource in `__enter__` and release it in `__exit__`.

In [3]:
class ManagedList:
    def __init__(self):
        self.data = []

    def __enter__(self):
        print('ENTER: acquiring list resource')
        self.data.append('resource-open')
        return self.data

    def __exit__(self, exc_type, exc, tb):
        print('EXIT: releasing list resource')
        self.data.append('resource-close')
        # Returning False means any exception is propagated.
        return False

with ManagedList() as items:
    items.append('doing-work')
    print('Inside context:', items)

print('After context:', items)

ENTER: acquiring list resource
Inside context: ['resource-open', 'doing-work']
EXIT: releasing list resource
After context: ['resource-open', 'doing-work', 'resource-close']


## 4. Create a Function-Based Context Manager with `contextlib.contextmanager`

Generator-based context managers are concise. Wrap the `yield` in `try/finally` for cleanup, or `try/except/finally` to also inspect or transform exceptions.

In [4]:
@contextlib.contextmanager
def temporary_text_file(prefix='ctx_', suffix='.txt'):
    temp = tempfile.NamedTemporaryFile(mode='w+', prefix=prefix, suffix=suffix, delete=False)
    path = Path(temp.name)
    try:
        temp.write('temporary payload')
        temp.flush()
        temp.close()
        yield path
    finally:
        if path.exists():
            path.unlink()

with temporary_text_file() as temp_path:
    print('Temp file exists inside block:', temp_path.exists())

print('Temp file exists after block:', temp_path.exists())

# Generator-based CM that catches and re-raises with context.
@contextlib.contextmanager
def wrap_io_error(label):
    try:
        yield
    except OSError as e:
        raise RuntimeError(f'[{label}] IO failed: {e}') from e

try:
    with wrap_io_error('loader'):
        raise OSError('disk read error')
except RuntimeError as e:
    print('Wrapped:', e)

Temp file exists inside block: True
Temp file exists after block: False
Wrapped: [loader] IO failed: disk read error


## 5. Handle Exceptions Inside Context Managers

`__exit__(exc_type, exc, tb)` can suppress exceptions by returning `True`, or propagate by returning `False`.

In [5]:
class SuppressValueError:
    def __enter__(self):
        print('enter SuppressValueError')
        return self

    def __exit__(self, exc_type, exc, tb):
        if exc_type is ValueError:
            print('suppressing ValueError:', exc)
            return True
        return False

class PropagateError:
    def __enter__(self):
        print('enter PropagateError')
        return self

    def __exit__(self, exc_type, exc, tb):
        print('propagating error if present')
        return False

with SuppressValueError():
    raise ValueError('handled inside __exit__')

try:
    with PropagateError():
        raise RuntimeError('this should bubble up')
except RuntimeError as e:
    print('caught outside:', e)

enter SuppressValueError
suppressing ValueError: handled inside __exit__
enter PropagateError
propagating error if present
caught outside: this should bubble up


## 6. Use `contextlib.suppress` and `contextlib.nullcontext`

`contextlib.suppress(*exceptions)` is a concise built-in alternative to a custom suppression class.  
`contextlib.nullcontext(value)` is a no-op context manager — useful when a context manager is optional (e.g., conditionally enabling a profiler or GPU autocast).

In [6]:
# contextlib.suppress — cleaner than writing a custom __exit__ suppressor.
with contextlib.suppress(FileNotFoundError):
    Path('/tmp/does_not_exist_xyz.txt').unlink()
print('suppress: no FileNotFoundError raised')

# Suppress multiple exception types at once.
with contextlib.suppress(FileNotFoundError, PermissionError):
    Path('/tmp/another_missing.txt').unlink()
print('suppress (multi): completed cleanly')

# contextlib.nullcontext — acts as an identity context manager.
def load_data(path, profiler=None):
    ctx = profiler if profiler is not None else contextlib.nullcontext()
    with ctx:
        return Path(path).read_text(encoding='utf-8')

content = load_data(SAMPLE_FILE)
print('nullcontext load result:', content.strip())

# Simulate the same call with a fake profiler to show it's interchangeable.
class FakeProfiler:
    def __enter__(self): print('profiler start'); return self
    def __exit__(self, *a): print('profiler stop'); return False

content_with_profiler = load_data(SAMPLE_FILE, profiler=FakeProfiler())
print('with profiler result:', content_with_profiler.strip())

suppress: no FileNotFoundError raised
suppress (multi): completed cleanly
nullcontext load result: line-1
line-2
profiler start
profiler stop
with profiler result: line-1
line-2


## 7. Build a Practical `Timer` Context Manager

A `Timer` is a common real-world context manager — used heavily in ML for benchmarking training loops, data loading, and inference.

In [7]:
class Timer:
    """Measures elapsed wall-clock time for a block."""

    def __init__(self, label=''):
        self.label = label
        self.elapsed = 0.0

    def __enter__(self):
        self._start = time.perf_counter()
        return self

    def __exit__(self, exc_type, exc, tb):
        self.elapsed = time.perf_counter() - self._start
        tag = f'[{self.label}] ' if self.label else ''
        print(f'{tag}elapsed: {self.elapsed * 1000:.3f} ms')
        return False

# Measure a simulated data-loading step.
with Timer('data-load') as t:
    time.sleep(0.05)
    items = list(range(1000))

print('Items loaded:', len(items))
print('Captured elapsed:', f'{t.elapsed * 1000:.3f} ms')

# Function-based equivalent using @contextmanager.
@contextlib.contextmanager
def timer(label=''):
    start = time.perf_counter()
    try:
        yield
    finally:
        elapsed = (time.perf_counter() - start) * 1000
        tag = f'[{label}] ' if label else ''
        print(f'{tag}elapsed: {elapsed:.3f} ms')

with timer('inference'):
    result = sum(x ** 2 for x in range(10_000))

[data-load] elapsed: 51.093 ms
Items loaded: 1000
Captured elapsed: 51.093 ms
[inference] elapsed: 3.405 ms


## 8. Compose Multiple Context Managers in One Statement

Resources are exited in reverse order of entry.

In [8]:
events = []

class Tracker:
    def __init__(self, name):
        self.name = name

    def __enter__(self):
        events.append(f'enter:{self.name}')
        return self

    def __exit__(self, exc_type, exc, tb):
        events.append(f'exit:{self.name}')
        return False

with Tracker('A') as a, Tracker('B') as b:
    events.append('inside')

print(events)
# Expected exit order: B then A

['enter:A', 'enter:B', 'inside', 'exit:B', 'exit:A']


## 9. Use `ExitStack` for Dynamic Resource Management

`ExitStack` is useful when the number of resources is known only at runtime.

In [9]:
dynamic_paths = []

for i in range(3):
    p = WORKSPACE / f'dynamic_{i}.txt'
    p.write_text(f'value-{i}', encoding='utf-8')
    dynamic_paths.append(p)

with contextlib.ExitStack() as stack:
    handles = [stack.enter_context(p.open('r', encoding='utf-8')) for p in dynamic_paths]
    values = [h.read() for h in handles]

print('Loaded values:', values)
print('All closed after ExitStack:', all(h.closed for h in handles))

Loaded values: ['value-0', 'value-1', 'value-2']
All closed after ExitStack: True


## 10. Write Small Tests for Context Manager Behavior

Use simple assertions to validate cleanup and exception behavior.

In [10]:
# Built-in file context manager closes file.
with (WORKSPACE / 'assert_file.txt').open('w', encoding='utf-8') as fh:
    fh.write('ok')
assert fh.closed is True

# Reverse exit order for composed managers.
assert events == ['enter:A', 'enter:B', 'inside', 'exit:B', 'exit:A']

# Function-based context manager cleaned up temp file.
assert temp_path.exists() is False

# ExitStack closed all tracked handles.
assert all(h.closed for h in handles)

# Suppression behavior check.
with SuppressValueError():
    raise ValueError('should be suppressed in test')

# contextlib.suppress does not raise on a missing file.
with contextlib.suppress(FileNotFoundError):
    Path('/tmp/nonexistent_assert_check.txt').unlink()

# nullcontext passes the enter value through.
with contextlib.nullcontext('sentinel') as v:
    assert v == 'sentinel'

# Timer records positive elapsed time.
with Timer('assert-timer') as t:
    time.sleep(0.01)
assert t.elapsed > 0

print('All lightweight context manager tests passed.')

enter SuppressValueError
suppressing ValueError: should be suppressed in test
[assert-timer] elapsed: 10.137 ms
All lightweight context manager tests passed.


### Best Practices

- Prefer `with` for any object that owns a resource (files, locks, sockets, DB sessions).
- Keep context manager scope small so cleanup happens early.
- In class-based managers, only suppress exceptions intentionally — returning `True` from `__exit__` silences all exceptions of that type.
- Use `contextlib.contextmanager` for concise one-off managers; wrap `yield` in `try/finally` for guaranteed cleanup.
- Use `contextlib.suppress` instead of a bare `try/except/pass` block for known ignorable exceptions.
- Use `contextlib.nullcontext` to make context managers optional (e.g., conditionally enabling a profiler or GPU autocast) without branching the call site.
- Use `ExitStack` when the number of resources is determined at runtime.
- Prefer a `Timer` context manager over manual `time.perf_counter()` calls to keep benchmarking code readable.